# 06 · High-resolution attention close-ups (ViT-S/16 + ABMIL)

Same 6 test slides as `fig_heatmaps` (one per subtype + one error). Per slide: overview heatmap · high-res zoom around
the most-attended spot · the 3 most-attended tiles at ~10x (spread apart) · the least-attended tissue tile.

### Inputs (Add Input)
1. `notebook4297f7f927` (Your Work → Notebooks) — features + tile positions
2. `notebookb9c62d68fa` (Your Work → Notebooks) — trained models
3. **UBC-OCEAN** (Competitions) — full-resolution slides

**Settings:** Accelerator **None** (CPU) · Internet **On** · **Run All** (~15–30 min, wait for the data banner first) or commit.
Download `closeups_bundle.zip`.

In [ ]:
# 1 · libraries
import time; T0 = time.time()
!pip install -q pyvips-binary pyvips
import os, sys, glob, re
os.makedirs('/tmp/code/mil', exist_ok=True)

In [ ]:
%%writefile /tmp/code/mil/__init__.py
"""MIL aggregators, training and evaluation for UBC-OCEAN subtype classification."""

In [ ]:
%%writefile /tmp/code/mil/models.py
"""Slide-level MIL aggregators.

Every model takes one bag of patch features ``x`` with shape (N, D) and returns a dict:
    logits    (C,)      class scores for the slide
    attn      (N,)      per-patch importance (for heatmaps), or None
    aux_loss  scalar    extra training loss (DSMIL instance loss, CLAM clustering loss), or None

Optional extra key:
    loss_logits (C,)    logits to use for the main training loss instead of ``logits`` (DSMIL)

Models with an auxiliary loss define ``aux_weight``: total = (1 - w) * main + w * aux.
Models that need the slide label for their auxiliary loss (DSMIL, CLAM) accept ``label=``;
the rest ignore it. Build models with ``build_model(name, in_dim, num_classes)``.
"""
import math

import torch
import torch.nn.functional as F
from torch import nn


class PoolMIL(nn.Module):
    """Per-patch MLP, then max or mean pooling, then a linear classifier.

    Layer names match the old capstone checkpoints (encoder.0, encoder.3, classifier),
    so those weights can still be loaded.
    """

    def __init__(self, in_dim, num_classes, pool='max', dropout=0.25):
        super().__init__()
        assert pool in ('max', 'mean')
        self.pool = pool
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 128), nn.ReLU(),
        )
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x, label=None):
        h = self.encoder(x)                                   # (N, 128)
        if self.pool == 'max':
            pooled, idx = h.max(dim=0)
            # importance = how many of the 128 features each patch "won"
            attn = torch.bincount(idx, minlength=len(x)).float() / h.shape[1]
        else:
            pooled, attn = h.mean(dim=0), None
        return dict(logits=self.classifier(pooled), attn=attn, aux_loss=None)


class GatedAttention(nn.Module):
    """Gated attention scores (Ilse et al. 2018). Returns raw scores (N, n_out)."""

    def __init__(self, dim, hidden, n_out=1, dropout=0.25):
        super().__init__()
        self.a = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Dropout(dropout))
        self.b = nn.Sequential(nn.Linear(dim, hidden), nn.Sigmoid(), nn.Dropout(dropout))
        self.c = nn.Linear(hidden, n_out)

    def forward(self, h):
        return self.c(self.a(h) * self.b(h))


class ABMIL(nn.Module):
    """Gated attention MIL (Ilse et al. 2018).

    Fixes old bug B1: attention is normalized over PATCHES (dim 0 of an (N, 1) tensor),
    not over a size-1 batch axis.
    """

    def __init__(self, in_dim, num_classes, hidden=256, dropout=0.25):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout))
        self.attention = GatedAttention(hidden, hidden // 2, 1, dropout)
        self.classifier = nn.Linear(hidden, num_classes)

    def forward(self, x, label=None):
        h = self.fc(x)                                        # (N, H)
        a = torch.softmax(self.attention(h).squeeze(1), 0)    # (N,)
        m = a @ h                                             # (H,)
        return dict(logits=self.classifier(m), attn=a, aux_loss=None)


class DSMIL(nn.Module):
    """Dual-stream MIL (Li, Li & Eliceiri, CVPR 2021). Cleaned port of the capstone version.

    Stream 1 scores each patch; the top patch per class is the "critical instance".
    Stream 2 attends to all patches by similarity to the critical instances.
    Prediction = mean of both streams; aux_loss = max-instance cross-entropy.
    """

    aux_weight = 0.5

    def __init__(self, in_dim, num_classes, hidden=128, dropout=0.25):
        super().__init__()
        self.i_classifier = nn.Linear(in_dim, num_classes)
        self.q = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.Tanh())
        self.v = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_dim, in_dim), nn.ReLU())
        self.b_classifier = nn.Conv1d(num_classes, num_classes, kernel_size=in_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, label=None):
        inst_logits = self.i_classifier(x)                    # (N, C)
        q, v = self.q(x), self.v(x)                           # (N, h), (N, D)
        crit = inst_logits.argmax(dim=0)                      # (C,) top patch per class
        attn = torch.softmax(q @ q[crit].T / math.sqrt(q.shape[1]), dim=0)  # (N, C)
        bag_feat = self.dropout(attn.T @ v)                   # (C, D)
        bag_logits = self.b_classifier(bag_feat.unsqueeze(0)).squeeze()      # (C,)
        max_inst = inst_logits.max(dim=0).values              # (C,)

        aux = None
        if label is not None:
            aux = F.cross_entropy(max_inst.unsqueeze(0), label.view(1))
        # Prediction = mean of both streams' probabilities (as log-probs, so softmax(logits) == probs).
        # Training loss = 0.5 * CE(bag_logits) + 0.5 * CE(max_inst), as in the original DSMIL.
        probs = 0.5 * (bag_logits.softmax(0) + max_inst.softmax(0))
        pred = probs.argmax()
        return dict(logits=probs.clamp_min(1e-8).log(), attn=attn[:, pred], aux_loss=aux,
                    loss_logits=bag_logits)


class CLAM_SB(nn.Module):
    """CLAM single-branch (Lu et al., Nat. Biomed. Eng. 2021).

    Gated attention pooling plus an instance-level clustering loss: the k most-attended
    patches are pushed to "positive" for the true class and "negative" for other classes;
    the k least-attended are "negative" for the true class.
    """

    aux_weight = 0.3   # official CLAM: 0.7 * bag loss + 0.3 * instance loss

    def __init__(self, in_dim, num_classes, hidden=512, attn_hidden=256, dropout=0.25, k_sample=8):
        super().__init__()
        self.k_sample = k_sample
        self.num_classes = num_classes
        self.fc = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout))
        self.attention = GatedAttention(hidden, attn_hidden, 1, dropout)
        self.classifier = nn.Linear(hidden, num_classes)
        self.instance_classifiers = nn.ModuleList(nn.Linear(hidden, 2) for _ in range(num_classes))

    def _inst_loss(self, a, h, c, in_class):
        k = min(self.k_sample, len(h) // 2 if in_class else len(h))
        if k < 1:
            return h.new_zeros(())
        top = h[torch.topk(a, k).indices]
        if in_class:
            bottom = h[torch.topk(-a, k).indices]
            feats = torch.cat([top, bottom])
            targets = torch.cat([torch.ones(k), torch.zeros(k)]).long().to(h.device)
        else:
            feats, targets = top, torch.zeros(k, dtype=torch.long, device=h.device)
        return F.cross_entropy(self.instance_classifiers[c](feats), targets)

    def forward(self, x, label=None):
        h = self.fc(x)                                        # (N, H)
        a = torch.softmax(self.attention(h).squeeze(1), 0)    # (N,)
        logits = self.classifier(a @ h)

        aux = None
        if label is not None:
            y = int(label)
            losses = [self._inst_loss(a, h, c, in_class=(c == y)) for c in range(self.num_classes)]
            aux = torch.stack(losses).mean()
        return dict(logits=logits, attn=a, aux_loss=aux)


class PPEG(nn.Module):
    """Pyramid position encoding generator from TransMIL."""

    def __init__(self, dim):
        super().__init__()
        self.c7 = nn.Conv2d(dim, dim, 7, 1, 3, groups=dim)
        self.c5 = nn.Conv2d(dim, dim, 5, 1, 2, groups=dim)
        self.c3 = nn.Conv2d(dim, dim, 3, 1, 1, groups=dim)

    def forward(self, x, side):
        cls, feat = x[:, :1], x[:, 1:]
        grid = feat.transpose(1, 2).reshape(x.shape[0], -1, side, side)
        grid = grid + self.c7(grid) + self.c5(grid) + self.c3(grid)
        return torch.cat([cls, grid.flatten(2).transpose(1, 2)], dim=1)


class TransLayer(nn.Module):
    def __init__(self, dim, dropout=0.1):
        super().__init__()
        from nystrom_attention import NystromAttention
        self.norm = nn.LayerNorm(dim)
        self.attn = NystromAttention(dim=dim, dim_head=dim // 8, heads=8, num_landmarks=dim // 2,
                                     pinv_iterations=6, residual=True, dropout=dropout)

    def forward(self, x):
        return x + self.attn(self.norm(x))

    @torch.no_grad()
    def cls_attention(self, x):
        """Exact softmax attention from token 0 (class token) to every token, mean over heads.

        O(N) memory, unlike asking NystromAttention for its full N×N matrix.
        """
        q, k, _ = self.attn.to_qkv(self.norm(x)).chunk(3, dim=-1)       # (1, L, heads*dh)
        heads = self.attn.heads
        q = q[0, :1].view(1, heads, -1).transpose(0, 1)                  # (heads, 1, dh)
        k = k[0].view(x.shape[1], heads, -1).transpose(0, 1)             # (heads, L, dh)
        scores = (q * self.attn.scale) @ k.transpose(1, 2)               # (heads, 1, L)
        return scores.softmax(-1).mean(0)[0]                             # (L,)


class TransMIL(nn.Module):
    """TransMIL (Shao et al., NeurIPS 2021): Nystrom self-attention between patches + PPEG.

    attn = attention from the class token to each patch in the last layer (mean over heads).
    """

    def __init__(self, in_dim, num_classes, hidden=512, dropout=0.25):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout))
        self.cls_token = nn.Parameter(torch.randn(1, 1, hidden))
        self.layer1 = TransLayer(hidden)
        self.ppeg = PPEG(hidden)
        self.layer2 = TransLayer(hidden)
        self.norm = nn.LayerNorm(hidden)
        self.classifier = nn.Linear(hidden, num_classes)

    def forward(self, x, label=None):
        n = len(x)
        h = self.fc(x).unsqueeze(0)                           # (1, N, H)
        side = math.ceil(math.sqrt(n))
        pad = side * side - n
        if pad:                                               # fill the square grid by repeating patches
            h = torch.cat([h, h[:, :pad]], dim=1)
        h = torch.cat([self.cls_token, h], dim=1)             # (1, 1 + side², H)
        h = self.layer1(h)
        h = self.ppeg(h, side)
        attn = None if self.training else self.layer2.cls_attention(h)[1:n + 1]   # (N,) real patches only
        h = self.layer2(h)
        logits = self.classifier(self.norm(h)[0, 0])
        return dict(logits=logits, attn=attn, aux_loss=None)


MODELS = {
    'maxpool': lambda d, c, **kw: PoolMIL(d, c, pool='max', **kw),
    'meanpool': lambda d, c, **kw: PoolMIL(d, c, pool='mean', **kw),
    'abmil': ABMIL,
    'dsmil': DSMIL,
    'clam_sb': CLAM_SB,
    'transmil': TransMIL,
}


def build_model(name, in_dim, num_classes, **kwargs):
    """kwargs: model hyperparameters, e.g. hidden=256, dropout=0.25 (None values are ignored)."""
    if name not in MODELS:
        raise ValueError(f'unknown model {name!r}; choose from {sorted(MODELS)}')
    return MODELS[name](in_dim, num_classes, **{k: v for k, v in kwargs.items() if v is not None})

In [ ]:
%%writefile /tmp/code/kfigs.py
"""Kaggle-side figures: t-SNE of encoder features, ABMIL attention heatmaps, attention vs tumour-mask agreement.

Used by kaggle/04_figures.ipynb (embedded there). All paths are passed in, so it also runs locally on fake data.
"""
import glob
import os
import re

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch

CLASSES = ['CC', 'EC', 'HGSC', 'LGSC', 'MC']
CAT = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4']          # validated categorical order (dataviz skill)
HEAT = LinearSegmentedColormap.from_list('heat', ['#fde3d6', '#f4a27f', '#eb6834', '#b8461b', '#6e2405'])
INK, INK2, GRID = '#0b0b0b', '#52514e', '#e4e3df'
EN = {'vits16_lunit': 'ViT-S/16 (Lunit)', 'ctranspath': 'CTransPath', 'phikon_v2': 'Phikon-v2', 'plip': 'PLIP'}
PATCH, SHRINK = 256, {False: 2, True: 4}

plt.rcParams.update({'font.size': 8.5, 'axes.titlesize': 9, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.edgecolor': '#9a9893', 'axes.linewidth': 0.6, 'xtick.color': INK2, 'ytick.color': INK2,
                     'pdf.fonttype': 42, 'savefig.bbox': 'tight', 'savefig.pad_inches': 0.03})


def save(fig, out, name):
    os.makedirs(out, exist_ok=True)
    for ext in ('pdf', 'png'):
        fig.savefig(os.path.join(out, f'{name}.{ext}'), dpi=250)
    plt.close(fig)
    print('wrote', name, flush=True)


# ----------------------------------------------------------------------------- t-SNE
def tsne_figure(features_root, labels, out, encoders, n_patches=1000, seed=0):
    """labels: DataFrame image_id, label (int). Slide-level (mean of tiles) and patch-level (n_patches sampled) t-SNE."""
    from sklearn.decomposition import PCA
    from sklearn.linear_model import LogisticRegression
    from sklearn.manifold import TSNE
    from sklearn.metrics import silhouette_score
    from sklearn.model_selection import StratifiedKFold, cross_val_predict
    from sklearn.metrics import balanced_accuracy_score
    from sklearn.preprocessing import StandardScaler

    rng = np.random.default_rng(seed)
    ids, y = labels.image_id.to_numpy(), labels.label.to_numpy()
    # same 1000 (slide, tile) picks for every encoder: stratified by subtype, 200 per class
    per_class = n_patches // len(CLASSES)
    picks = []
    for c in range(len(CLASSES)):
        cls_ids = ids[y == c]
        for s in rng.choice(cls_ids, per_class, replace=True):
            picks.append(s)
    picks = np.array(picks)
    rows, scores = [], []
    fig, axes = plt.subplots(2, len(encoders), figsize=(2.05 * len(encoders), 4.4))
    axes = np.atleast_2d(axes).reshape(2, -1)
    for j, enc in enumerate(encoders):
        means, patch_feats, patch_y = [], [], []
        tile_choice = {}
        for s in ids:
            f = np.load(os.path.join(features_root, enc, f'{s}.npy')).astype(np.float32)
            means.append(f.mean(0))
        for k, s in enumerate(picks):
            f = np.load(os.path.join(features_root, enc, f'{s}.npy'), mmap_mode='r')
            t = tile_choice.setdefault((k, s), int(np.random.default_rng(seed + k).integers(len(f))))
            patch_feats.append(np.asarray(f[t], dtype=np.float32))
            patch_y.append(y[ids == s][0])
        for row, (X, yy, name) in enumerate([(np.stack(means), y, 'slide'), (np.stack(patch_feats), np.array(patch_y), 'patch')]):
            Xs = StandardScaler().fit_transform(X)
            Xp = PCA(n_components=min(50, Xs.shape[1], len(Xs) - 1), random_state=seed).fit_transform(Xs)
            sil = silhouette_score(Xp, yy)
            probe = ''
            if name == 'slide':   # linear probe: 5-fold CV logistic regression on slide-mean features
                pred = cross_val_predict(LogisticRegression(max_iter=2000, C=0.5, class_weight='balanced'), Xs, yy,
                                         cv=StratifiedKFold(5, shuffle=True, random_state=seed))
                probe = balanced_accuracy_score(yy, pred)
            emb = TSNE(n_components=2, perplexity=30 if len(Xp) > 100 else 10, random_state=seed, init='pca').fit_transform(Xp)
            ax = axes[row, j]
            for c in range(len(CLASSES)):
                m = yy == c
                ax.scatter(emb[m, 0], emb[m, 1], s=6 if name == 'slide' else 3, color=CAT[c], linewidths=0, alpha=0.85, label=CLASSES[c])
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_visible(False)
            sub = f'silhouette {sil:.3f}' + (f'\nlinear probe bacc {probe:.3f}' if probe != '' else '')
            ax.set_title((f'{EN.get(enc, enc)}\n' if row == 0 else '') + sub, fontsize=7.5, color=INK, loc='left')
            scores.append(dict(encoder=enc, level=name, n=len(yy), silhouette=sil, linear_probe_bacc=probe if probe != '' else np.nan))
        print(f'[tsne] {enc} done', flush=True)
    axes[0, 0].set_ylabel('slide level\n(mean of tiles, n = %d)' % len(ids), fontsize=7.5, color=INK2)
    axes[1, 0].set_ylabel('patch level\n(%d tiles)' % len(picks), fontsize=7.5, color=INK2)
    handles = [Patch(color=CAT[c], label=CLASSES[c]) for c in range(len(CLASSES))]
    fig.legend(handles=handles, ncol=5, frameon=False, loc='upper center', bbox_to_anchor=(0.5, 1.03), fontsize=7.5)
    save(fig, out, 'fig_tsne')
    pd.DataFrame(scores).to_csv(os.path.join(out, 'tsne_scores.csv'), index=False)
    return pd.DataFrame(scores)


# ----------------------------------------------------------------------------- attention
def load_models(runs_dir, build_model, in_dim, device):
    """{(seed, fold): model} for every best.pt under runs_dir/seed*/fold*/"""
    models = {}
    for p in sorted(glob.glob(os.path.join(runs_dir, 'seed*', 'fold*', 'best.pt'))):
        seed = int(re.search(r'seed(\d+)', p).group(1)); fold = int(re.search(r'fold(\d+)', p).group(1))
        ck = torch.load(p, map_location=device, weights_only=False)
        m = build_model(ck.get('model_name', 'abmil'), in_dim, len(CLASSES), **ck.get('model_kwargs', {})).to(device).eval()
        m.load_state_dict(ck['model'])
        models[(seed, fold)] = m
    return models


@torch.no_grad()
def slide_attention(models, keys, x):
    """Average over the given models: probabilities (C,) and attention as per-model percentile ranks (N,)."""
    probs, ranks = [], []
    for k in keys:
        out = models[k](x)
        probs.append(out['logits'].float().softmax(0).cpu().numpy())
        a = out['attn'].float().cpu().numpy()
        ranks.append(a.argsort().argsort() / max(1, len(a) - 1))     # rank-normalise so every model counts equally
    return np.mean(probs, 0), np.mean(ranks, 0)


def tile_boxes(coords, is_tma, scale):
    """tile rectangles (x0, y0, size) in a canvas that is `scale` x the original slide resolution"""
    sh = SHRINK[bool(is_tma)]
    return coords[:, 1] * sh * scale, coords[:, 0] * sh * scale, PATCH * sh * scale


def paint(canvas_hw, coords, is_tma, scale, values):
    canvas = np.full(canvas_hw, np.nan, dtype=np.float32)
    xs, ys, size = tile_boxes(coords, is_tma, scale)
    for x0, y0, v in zip(xs, ys, values):
        canvas[int(y0):int(np.ceil(y0 + size)), int(x0):int(np.ceil(x0 + size))] = v
    return canvas


def load_image_scaled(path, target_w):
    """RGB uint8 array resized to target_w (keeps aspect) via pyvips (streams big PNGs)."""
    import pyvips
    img = pyvips.Image.thumbnail(path, target_w, height=10 ** 7)
    if img.bands > 3:
        img = img.extract_band(0, n=3)
    return img.numpy()


def mask_classes(mask_rgb):
    """per pixel: 1 tumour (red), 2 other annotated tissue (green stroma / blue necrosis), 0 unannotated"""
    r, g, b = [mask_rgb[..., i].astype(int) for i in range(3)]
    out = np.zeros(mask_rgb.shape[:2], np.uint8)
    out[(g > 128) & (r < 128) | (b > 128) & (r < 128)] = 2
    out[(r > 128) & (g < 128) & (b < 128)] = 1
    return out


def tile_mask_labels(cls_map, coords, is_tma, scale):
    """per tile: 1 tumour (>=50% tumour px), 0 non-tumour (>=50% annotated non-tumour, <10% tumour), -1 otherwise"""
    xs, ys, size = tile_boxes(coords, is_tma, scale)
    lab = np.full(len(coords), -1, np.int8)
    for i, (x0, y0) in enumerate(zip(xs, ys)):
        patch = cls_map[int(y0):int(np.ceil(y0 + size)), int(x0):int(np.ceil(x0 + size))]
        if patch.size == 0:
            continue
        tumour, other = (patch == 1).mean(), (patch == 2).mean()
        if tumour >= 0.5:
            lab[i] = 1
        elif other >= 0.5 and tumour < 0.1:
            lab[i] = 0
    return lab


def mask_agreement(models, slides, features_dir, coords_dir, mask_files, device, out, canvas_w=1024):
    """For every slide with a mask: AUROC of attention for tumour vs non-tumour tiles, using only models that never
    trained on that slide (test slides: all models; CV slides: the models whose validation fold contained it)."""
    from sklearn.metrics import roc_auc_score
    rows = []
    for _, s in slides[slides.image_id.isin(mask_files)].iterrows():
        keys = [k for k in models if s.fold == 0 or k[1] == s.fold]
        if not keys:
            continue
        x = torch.from_numpy(np.load(os.path.join(features_dir, f'{s.image_id}.npy'))).float().to(device)
        coords = np.load(os.path.join(coords_dir, f'{s.image_id}.npy'))
        _, att = slide_attention(models, keys, x)
        mask = load_image_scaled(mask_files[s.image_id], canvas_w)
        scale = mask.shape[1] / s.image_width
        lab = tile_mask_labels(mask_classes(mask), coords, s.is_tma, scale)
        t, n = att[lab == 1], att[lab == 0]
        row = dict(image_id=s.image_id, label=CLASSES[s.label], split='test' if s.fold == 0 else f'cv{s.fold}',
                   n_models=len(keys), n_tumour_tiles=len(t), n_nontumour_tiles=len(n))
        if len(t) >= 5 and len(n) >= 5:
            row['auroc'] = roc_auc_score(np.r_[np.ones(len(t)), np.zeros(len(n))], np.r_[t, n])
            top = att >= np.quantile(att, 0.9)
            known = lab >= 0
            row['top10_tumour_frac'] = (lab[top & known] == 1).mean() if (top & known).any() else np.nan
            row['base_tumour_frac'] = (lab[known] == 1).mean()
        rows.append(row)
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(out, 'heatmap_mask_agreement.csv'), index=False)
    return df


def load_slide_10x(path, is_tma):
    """Slide at the resolution the tiles were cut from (WSI 20x/2, TMA 40x/4 = ~10x), RGB uint8."""
    import pyvips
    img = pyvips.Image.new_from_file(path, access='sequential')
    if img.bands > 3:
        img = img.extract_band(0, n=3)
    sh = SHRINK[bool(is_tma)]
    return img.shrink(sh, sh).numpy()


def spread_top(att, coords, k, min_dist=3 * PATCH):
    """indices of the k highest-attention tiles, at least min_dist apart (10x pixels) so they show different spots"""
    chosen = []
    for i in np.argsort(-att):
        if all(np.abs(coords[i] - coords[j]).max() >= min_dist for j in chosen):
            chosen.append(i)
        if len(chosen) == k:
            break
    return chosen


def closeup_figure(models, picks, features_dir, coords_dir, thumb_files, full_files, device, out, n_top=3, zoom_tiles=6):
    """Per slide: overview heatmap | high-res zoom with attention | top-n attended tiles (10x) | lowest-attention tile."""
    ncol = 3 + n_top
    fig, axes = plt.subplots(len(picks), ncol, figsize=(1.95 * ncol, 2.05 * len(picks)))
    axes = np.atleast_2d(axes)
    for r, (_, s) in enumerate(picks.iterrows()):
        keys = [k for k in models if s.fold == 0 or k[1] == s.fold]
        x = torch.from_numpy(np.load(os.path.join(features_dir, f'{s.image_id}.npy'))).float().to(device)
        coords = np.load(os.path.join(coords_dir, f'{s.image_id}.npy'))
        prob, att = slide_attention(models, keys, x)
        arr = load_slide_10x(full_files[s.image_id], s.is_tma)
        top = spread_top(att, coords, n_top)
        low = int(np.argmin(att))
        # zoom window (10x pixels) centred on the top tile
        half = zoom_tiles * PATCH // 2
        cy, cx = coords[top[0]] + PATCH // 2
        y0 = int(np.clip(cy - half, 0, max(0, arr.shape[0] - 2 * half))); x0 = int(np.clip(cx - half, 0, max(0, arr.shape[1] - 2 * half)))
        crop = arr[y0:y0 + 2 * half, x0:x0 + 2 * half]
        zheat = np.full(crop.shape[:2], np.nan, np.float32)
        for (ty, tx), v in zip(coords, att):
            ys, xs = ty - y0, tx - x0
            if -PATCH < ys < crop.shape[0] and -PATCH < xs < crop.shape[1]:
                zheat[max(0, ys):max(0, ys + PATCH), max(0, xs):max(0, xs + PATCH)] = v
        # overview
        ov = load_image_scaled(thumb_files[s.image_id], 600) if s.image_id in thumb_files else arr[::max(1, arr.shape[1] // 600), ::max(1, arr.shape[1] // 600)]
        scale = ov.shape[1] / s.image_width
        ax = axes[r, 0]
        ax.imshow(ov)
        ax.imshow(np.ma.masked_invalid(paint(ov.shape[:2], coords, s.is_tma, scale, att)), cmap=HEAT, vmin=0, vmax=1, alpha=0.6, interpolation='nearest')
        sh = SHRINK[bool(s.is_tma)]
        ax.add_patch(plt.Rectangle((x0 * sh * scale, y0 * sh * scale), 2 * half * sh * scale, 2 * half * sh * scale, fill=False, ec=INK, lw=1.2))
        pred = int(prob.argmax())
        ax.set_title(f'{s.image_id} · true {CLASSES[s.label]}\npred {CLASSES[pred]} ({prob[pred]:.2f}) {"✓" if pred == s.label else "✗"}',
                     fontsize=7, loc='left', color=INK)
        # zoom with attention + numbered top tiles
        ax = axes[r, 1]
        ax.imshow(crop)
        ax.imshow(np.ma.masked_invalid(zheat), cmap=HEAT, vmin=0, vmax=1, alpha=0.35, interpolation='nearest')
        for n, i in enumerate(top, 1):
            ty, tx = coords[i][0] - y0, coords[i][1] - x0
            if 0 <= ty < crop.shape[0] and 0 <= tx < crop.shape[1]:
                ax.add_patch(plt.Rectangle((tx, ty), PATCH, PATCH, fill=False, ec=INK, lw=1.2))
                ax.text(tx + 8, ty + 40, str(n), color='white', fontsize=7, fontweight='bold')
        ax.set_title(f'zoom ({2 * half} px at ~10x)', fontsize=7, loc='left', color=INK2)
        # tiles
        panels = [(2 + n, i, f'#{n + 1} attention {att[i]:.2f}', INK) for n, i in enumerate(top)]
        panels.append((ncol - 1, low, f'lowest {att[low]:.2f}', INK2))      # always the last column
        for c, i, label, color in panels:
            ty, tx = coords[i]
            axes[r, c].imshow(arr[ty:ty + PATCH, tx:tx + PATCH])
            axes[r, c].set_title(label, fontsize=7, loc='left', color=color)
        for ax in axes[r]:
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_visible(False)
        del arr
        print(f'[closeup] {s.image_id} done', flush=True)
    save(fig, out, 'fig_heatmap_closeups')


def heatmap_figure(models, picks, features_dir, coords_dir, image_files, mask_files, device, out, canvas_w=900):
    """One column per picked slide: tissue | attention overlay | (mask if available)."""
    has_mask = any(p.image_id in mask_files for _, p in picks.iterrows())
    nrow = 3 if has_mask else 2
    fig, axes = plt.subplots(nrow, len(picks), figsize=(2.0 * len(picks), 2.1 * nrow))
    axes = np.atleast_2d(axes).reshape(nrow, -1)
    for j, (_, s) in enumerate(picks.iterrows()):
        keys = [k for k in models if s.fold == 0 or k[1] == s.fold]
        x = torch.from_numpy(np.load(os.path.join(features_dir, f'{s.image_id}.npy'))).float().to(device)
        coords = np.load(os.path.join(coords_dir, f'{s.image_id}.npy'))
        prob, att = slide_attention(models, keys, x)
        if s.image_id in image_files:
            img = load_image_scaled(image_files[s.image_id], canvas_w)
        else:   # no slide image attached: plain canvas
            h = int(round(canvas_w * s.image_height / s.image_width))
            img = np.full((h, canvas_w, 3), 245, np.uint8)
        scale = img.shape[1] / s.image_width
        heat = paint(img.shape[:2], coords, s.is_tma, scale, att)
        pred = int(prob.argmax())
        axes[0, j].imshow(img)
        axes[1, j].imshow(img)
        axes[1, j].imshow(np.ma.masked_invalid(heat), cmap=HEAT, vmin=0, vmax=1, alpha=0.65, interpolation='nearest')
        ok = '✓' if pred == s.label else '✗'
        axes[0, j].set_title(f'{s.image_id} · true {CLASSES[s.label]}\npred {CLASSES[pred]} ({prob[pred]:.2f}) {ok}', fontsize=7.2,
                             color=INK, loc='left')
        if has_mask:
            if s.image_id in mask_files:
                m = load_image_scaled(mask_files[s.image_id], img.shape[1])
                axes[2, j].imshow(img)
                axes[2, j].imshow(np.ma.masked_where(mask_classes(m) == 0, mask_classes(m)),
                                  cmap=LinearSegmentedColormap.from_list('m', ['#e34948', '#1baf7a']), vmin=1, vmax=2, alpha=0.5)
            else:
                axes[2, j].text(0.5, 0.5, 'no mask', ha='center', va='center', color=INK2, transform=axes[2, j].transAxes)
        for i in range(nrow):
            axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
            for sp in axes[i, j].spines.values():
                sp.set_visible(False)
    axes[0, 0].set_ylabel('tissue', color=INK2)
    axes[1, 0].set_ylabel('attention\n(dark = high)', color=INK2)
    if has_mask:
        axes[2, 0].set_ylabel('mask (red tumour,\ngreen other)', color=INK2)
    save(fig, out, 'fig_heatmaps')

In [ ]:
%%writefile /tmp/code/assignment.csv
file_name,label,is_tma,fold
10077,1,0,4
10143,2,0,0
1020,2,0,4
10246,1,0,5
10252,3,0,1
10469,1,0,1
10548,0,0,1
10642,2,0,3
1080,2,0,1
10800,1,0,5
10896,2,0,4
1101,2,0,0
11263,0,0,2
11417,2,0,3
11431,1,0,5
11557,3,0,0
11559,2,0,3
12159,2,0,2
12222,0,0,5
12244,3,0,1
12442,0,0,5
1252,2,0,4
12522,1,0,2
1289,2,0,0
12902,1,0,4
1295,2,0,4
13364,2,0,0
13387,3,0,0
13526,0,0,3
13568,3,1,0
13987,0,0,4
14039,2,0,5
14051,0,0,5
14127,2,0,1
14312,4,0,2
14401,1,0,4
14424,1,0,0
14532,4,0,2
14542,2,0,3
14617,1,0,5
15139,2,0,0
15188,1,0,2
15209,2,0,2
15221,2,0,1
15231,0,0,0
15293,2,0,5
15470,0,0,0
15486,0,0,2
15583,3,0,2
15671,1,0,4
15742,1,0,2
15871,2,0,1
15912,1,0,0
16042,2,0,2
16064,1,0,1
16209,2,0,4
16325,0,0,0
16494,2,0,4
1660,0,0,4
1666,2,0,0
16876,0,0,3
16986,2,0,0
17067,0,0,1
17174,2,0,5
17291,0,0,4
17365,2,0,5
17416,2,0,2
17487,1,0,4
17637,2,1,2
17738,1,0,5
1774,2,0,4
17854,2,0,5
18014,0,0,2
18138,1,0,2
18196,2,0,2
18547,2,0,2
18568,2,0,3
18607,1,0,0
18810,1,0,0
18813,0,0,2
18896,2,0,3
18914,2,0,4
18981,2,0,0
19030,1,0,0
19157,2,0,5
1925,2,0,2
19255,2,0,1
1943,0,0,0
19512,2,0,1
1952,0,0,0
19569,1,0,5
20205,0,0,0
20312,2,0,4
20316,2,0,1
20329,4,0,1
20463,2,0,4
20670,2,0,4
20858,2,0,2
20882,1,0,0
2097,2,0,5
21020,4,1,1
21232,1,0,3
21260,3,0,0
21303,3,0,4
21373,1,0,3
21432,2,0,1
21445,4,0,5
21910,1,0,3
21929,0,0,5
22155,0,0,5
22221,1,0,0
2227,4,0,0
22290,0,0,0
22425,0,0,1
22489,0,0,4
22654,2,0,2
22740,1,0,4
22924,1,0,3
23523,4,0,0
23629,1,0,0
23796,3,0,3
2391,3,0,0
24023,1,0,3
24507,2,0,4
24563,0,0,2
24617,2,0,0
24759,2,0,4
24991,2,0,5
25256,3,0,5
25331,4,0,4
25561,2,0,2
25604,1,0,4
25792,2,0,2
25923,0,0,3
25928,0,0,0
26025,3,0,4
26124,2,0,4
26190,2,0,5
26219,0,0,1
26533,0,0,1
26603,2,0,4
26644,2,0,0
2666,1,0,5
26862,3,0,1
26939,1,0,0
26950,0,0,3
2706,2,0,5
27245,2,0,2
27249,2,0,1
27315,0,0,4
27739,2,0,0
27747,2,0,4
27851,1,0,0
27950,2,0,4
28028,2,0,0
28066,1,0,3
281,3,0,5
28121,2,0,2
28393,2,0,1
28519,2,0,0
28562,4,0,3
286,1,0,2
28603,0,0,1
28736,1,0,5
28821,2,0,4
28922,2,0,1
2906,1,0,4
29084,3,1,5
29147,2,0,1
29200,2,0,1
29240,2,0,0
29331,2,0,3
29615,0,0,1
29888,2,0,2
29904,2,0,3
29915,0,0,5
30203,0,0,1
30272,0,0,1
30369,0,0,5
30508,2,0,5
30515,1,0,2
30539,2,0,3
3055,2,0,5
30712,1,0,4
30738,2,0,2
30792,1,0,3
30794,2,0,3
3084,4,0,2
30868,4,0,5
3092,3,0,4
3098,2,0,5
30986,4,0,2
31033,0,0,2
31297,4,0,1
31300,3,0,0
31333,2,0,0
31383,1,0,1
31473,2,0,0
31594,1,1,3
31793,3,0,5
3191,0,0,2
32032,0,0,1
32035,2,0,4
32042,1,0,1
32112,2,0,5
32192,2,0,1
3222,1,0,5
32432,1,0,2
32596,2,0,4
32636,1,0,5
3264,2,0,3
33708,3,0,1
33839,3,0,2
33976,3,0,0
33984,1,0,1
34247,2,0,1
34277,3,0,4
34508,4,0,1
34649,2,0,1
34688,3,0,4
34690,1,0,4
34720,2,0,1
34822,1,0,3
34845,2,0,5
3511,1,0,0
35197,0,0,5
35239,2,0,1
35565,4,1,5
35592,2,0,5
35652,2,0,0
35754,1,0,1
35792,4,0,4
35909,1,0,0
35953,1,0,1
35998,2,0,5
36008,2,0,2
36063,0,0,1
36204,2,0,2
36302,0,1,5
36499,2,0,0
36513,2,0,0
36583,3,1,3
36639,2,0,1
36678,4,0,2
3672,3,0,2
36783,4,1,2
37190,4,0,5
37307,4,0,4
37367,2,0,5
37385,3,1,4
37655,2,0,1
38018,2,0,1
38019,4,0,4
38041,2,0,3
38048,2,0,5
38097,2,0,2
38118,1,0,1
38349,0,0,0
38366,3,0,0
38479,1,0,0
38535,2,0,1
38585,3,0,5
38647,1,0,3
38669,0,0,4
38687,0,0,3
3881,2,0,2
38849,3,0,1
38901,1,0,5
38959,3,0,3
39144,2,0,1
39146,2,0,3
39172,2,0,0
39208,4,0,0
39252,4,0,2
39255,2,0,2
39258,3,0,2
39269,1,0,4
39297,2,0,2
39365,2,0,5
39425,2,0,0
39466,3,0,5
39728,2,0,4
39872,4,0,3
39880,0,0,4
39893,2,0,4
3997,4,0,0
39990,2,0,0
4,2,0,4
40079,1,0,2
40124,1,0,2
40129,0,0,2
40503,1,0,5
40639,2,0,0
40864,3,1,2
40888,0,0,5
41099,3,0,2
4134,4,1,0
41361,2,0,4
41368,1,1,5
41586,0,1,1
41801,2,0,4
4211,2,0,5
42125,0,0,4
42260,4,0,5
42296,0,0,0
42549,0,0,0
42857,0,1,0
431,2,0,0
43280,1,0,0
43390,2,0,0
43432,2,0,1
43671,2,0,3
43796,0,0,3
43815,2,0,3
43875,2,0,3
43998,1,0,0
44232,1,0,3
44283,0,0,2
44432,2,0,2
44530,4,0,1
44581,0,0,0
44603,0,1,2
44700,1,0,5
44804,1,0,4
44962,1,0,2
44976,0,0,2
45104,2,0,5
45185,2,0,2
45254,1,0,1
45578,1,0,1
45630,1,0,3
45725,0,0,5
45990,0,0,3
4608,1,0,2
46139,2,0,0
46172,2,0,0
46435,2,0,0
46444,1,0,4
46469,2,0,3
46543,0,0,2
46688,1,0,3
46736,2,0,2
46769,0,0,1
46793,2,0,0
46815,4,0,3
47020,2,0,5
47035,1,1,0
47105,1,0,2
47431,4,0,5
47837,2,0,5
47911,2,0,1
47960,0,0,3
4797,4,0,3
47984,2,0,0
4827,0,0,5
48502,4,0,4
48506,0,0,2
48550,4,0,4
48734,1,1,4
4877,0,0,2
48861,2,0,4
48973,2,0,2
49281,2,0,1
49587,4,0,0
4963,2,0,4
49872,3,0,4
49942,1,0,3
49995,1,0,3
50048,1,0,0
5015,2,0,5
50246,1,0,3
50304,1,0,0
50589,2,0,3
50712,2,0,3
50878,2,0,3
50932,2,1,4
50962,1,0,1
51021,2,0,2
51032,1,0,5
51128,1,0,0
5114,2,0,2
51215,1,0,2
51346,0,0,0
51499,1,0,3
51679,1,0,1
51832,2,0,3
51893,4,0,4
52108,1,0,4
52259,0,0,4
52275,2,0,5
52308,2,0,0
52375,0,0,4
52420,0,0,0
52438,2,0,2
52461,0,0,4
5251,2,0,1
52612,3,0,0
5264,2,0,3
5265,1,0,0
52752,1,0,4
52784,2,0,3
52836,1,0,5
52846,1,0,1
52931,0,0,1
53059,2,0,2
5307,2,0,4
53098,1,0,0
53377,0,0,3
53402,1,0,4
53655,1,1,1
53688,2,0,1
53859,1,0,5
53900,1,0,3
54007,2,0,1
54408,1,0,0
54473,3,0,4
54506,2,0,4
5456,4,0,3
54590,2,0,0
54825,2,0,4
54928,0,0,3
54949,2,0,0
54990,2,0,3
55279,1,0,1
55281,2,0,2
55287,0,0,5
55876,1,0,5
56117,4,0,0
56221,2,0,5
56351,4,0,2
56500,3,0,3
56799,4,0,3
56843,2,0,3
56861,0,0,1
56875,1,0,0
56947,0,0,3
56993,4,0,0
57100,2,0,2
57162,3,0,3
57265,4,0,5
57468,2,0,5
57598,0,0,0
57696,0,1,3
57711,1,0,2
5851,2,0,4
5852,3,0,1
58895,2,0,0
58947,0,0,3
58974,0,0,2
59002,1,0,3
59031,2,0,0
59383,2,0,2
59515,0,0,4
59589,2,0,3
5970,0,0,3
59760,1,0,2
59900,1,0,1
5992,2,0,5
60058,2,0,3
60287,2,0,0
60685,0,0,0
60928,1,0,2
60936,1,0,1
60988,0,0,5
61033,2,0,3
61089,2,0,1
61100,3,0,3
61280,2,0,5
61320,1,0,1
6140,2,0,5
61424,2,0,4
61493,2,0,3
61689,2,0,3
6175,2,0,3
61797,2,1,0
61823,1,0,4
61852,2,0,5
61961,0,0,0
62476,4,0,0
62641,2,0,0
6281,1,0,2
62828,0,0,1
63015,0,0,0
63121,1,0,3
63165,0,0,0
63289,0,0,5
63298,2,0,0
63367,1,0,5
63429,1,0,5
6359,0,0,4
6363,1,0,4
63836,1,0,1
63897,2,0,3
63941,2,0,1
64111,2,0,0
64188,2,0,1
6449,0,0,4
64629,2,0,1
64771,1,0,0
64824,0,0,3
64950,2,0,0
65022,3,0,3
65094,4,0,1
65300,2,0,2
65371,2,0,0
65533,2,0,2
6558,2,0,3
6582,4,0,3
66,3,0,1
6793,1,0,2
6843,2,0,2
6898,3,0,5
6951,2,0,3
706,2,0,0
7204,1,0,4
7329,1,0,0
7482,0,0,3
7490,1,0,2
7955,0,0,4
8130,3,0,5
8213,2,0,5
8240,2,0,4
8279,0,0,5
8280,2,1,3
8531,1,0,1
8713,2,0,4
8805,2,0,0
8985,3,0,3
91,2,1,1
9154,2,0,1
9183,3,0,2
9200,4,1,4
9254,4,0,1
9341,2,0,0
9509,2,0,5
9658,0,0,2
9697,2,0,5
970,2,0,1

In [ ]:
%%writefile /tmp/code/kaggle_train.csv
image_id,label,image_width,image_height,is_tma
4,HGSC,23785,20008,False
66,LGSC,48871,48195,False
91,HGSC,3388,3388,True
281,LGSC,42309,15545,False
286,EC,37204,30020,False
431,HGSC,39991,40943,False
706,HGSC,75606,25965,False
970,HGSC,32131,18935,False
1020,HGSC,36585,33751,False
1080,HGSC,31336,23200,False
1101,HGSC,26306,18403,False
1252,HGSC,60420,27480,False
1289,HGSC,43940,26785,False
1295,HGSC,48320,17700,False
1660,CC,83340,20447,False
1666,HGSC,69900,16083,False
1774,HGSC,44231,37571,False
1925,HGSC,44847,32951,False
1943,CC,73730,34949,False
1952,CC,33685,38053,False
2097,HGSC,31696,21984,False
2227,MC,60063,40187,False
2391,LGSC,58075,26192,False
2666,EC,53270,44031,False
2706,HGSC,71289,22569,False
2906,EC,77265,31059,False
3055,HGSC,73631,37323,False
3084,MC,44749,42274,False
3092,LGSC,38308,20600,False
3098,HGSC,68160,26751,False
3191,CC,10688,23348,False
3222,EC,31881,21623,False
3264,HGSC,63480,28313,False
3511,EC,29305,17826,False
3672,LGSC,62463,21527,False
3881,HGSC,28073,14116,False
3997,MC,49467,29610,False
4134,MC,2964,2964,True
4211,HGSC,35243,34431,False
4608,EC,33155,39867,False
4797,MC,53623,38587,False
4827,CC,57240,31509,False
4877,CC,57724,42954,False
4963,HGSC,40708,48923,False
5015,HGSC,48645,44254,False
5114,HGSC,33646,29171,False
5251,HGSC,105763,18704,False
5264,HGSC,45054,22207,False
5265,EC,63815,44795,False
5307,HGSC,71975,27683,False
5456,MC,45663,45226,False
5851,HGSC,59353,28393,False
5852,LGSC,71905,21963,False
5970,CC,27265,22900,False
5992,HGSC,70863,28702,False
6140,HGSC,42775,41227,False
6175,HGSC,60306,29620,False
6281,EC,75116,48293,False
6359,CC,39280,37289,False
6363,EC,69610,43670,False
6449,CC,81867,43860,False
6558,HGSC,47083,38630,False
6582,MC,40304,29208,False
6793,EC,69906,38027,False
6843,HGSC,84220,41951,False
6898,LGSC,38200,33146,False
6951,HGSC,44544,35909,False
7204,EC,25302,33623,False
7329,EC,26305,33760,False
7482,CC,44440,38590,False
7490,EC,59147,31826,False
7955,CC,63480,15945,False
8130,LGSC,66247,29130,False
8213,HGSC,36683,25710,False
8240,HGSC,33702,35086,False
8279,CC,90550,42022,False
8280,HGSC,2964,2964,True
8531,EC,51303,27787,False
8713,HGSC,70772,48629,False
8805,HGSC,32123,20520,False
8985,LGSC,65003,31754,False
9154,HGSC,16582,14705,False
9183,LGSC,74091,34185,False
9200,MC,3388,3388,True
9254,MC,59807,37445,False
9341,HGSC,69320,36671,False
9509,HGSC,21288,20626,False
9658,CC,52900,45380,False
9697,HGSC,42323,19928,False
10077,EC,53143,39150,False
10143,HGSC,40063,42207,False
10246,EC,48522,38690,False
10252,LGSC,49053,39794,False
10469,EC,42181,28660,False
10548,CC,79600,25645,False
10642,HGSC,69213,41391,False
10800,EC,39547,29240,False
10896,HGSC,48060,37792,False
11263,CC,80529,25660,False
11417,HGSC,32591,32600,False
11431,EC,45110,22862,False
11557,LGSC,19624,16602,False
11559,HGSC,43449,17806,False
12159,HGSC,33335,22684,False
12222,CC,63651,26220,False
12244,LGSC,27921,17683,False
12442,CC,85729,38439,False
12522,EC,46605,45511,False
12902,EC,46481,42007,False
13364,HGSC,16017,16352,False
13387,LGSC,27689,19307,False
13526,CC,72523,42491,False
13568,LGSC,2964,2964,True
13987,CC,65805,38010,False
14039,HGSC,27786,14400,False
14051,CC,67046,18674,False
14127,HGSC,47823,19006,False
14312,MC,66727,28600,False
14401,EC,63909,43603,False
14424,EC,68995,42311,False
14532,MC,41640,35386,False
14542,HGSC,79563,36089,False
14617,EC,24583,18405,False
15139,HGSC,74343,38730,False
15188,EC,53455,47347,False
15209,HGSC,49503,36931,False
15221,HGSC,42342,19582,False
15231,CC,100548,38277,False
15293,HGSC,49327,19044,False
15470,CC,64751,18986,False
15486,CC,53008,27243,False
15583,LGSC,32821,23837,False
15671,EC,36960,24947,False
15742,EC,61726,46995,False
15871,HGSC,33559,20877,False
15912,EC,59962,24346,False
16042,HGSC,77563,26032,False
16064,EC,42725,32651,False
16209,HGSC,55973,39506,False
16325,CC,55348,24880,False
16494,HGSC,48043,25247,False
16876,CC,24301,25740,False
16986,HGSC,33547,28025,False
17067,CC,62411,34260,False
17174,HGSC,31644,29109,False
17291,CC,52562,37932,False
17365,HGSC,43902,26689,False
17416,HGSC,78351,34531,False
17487,EC,33311,27967,False
17637,HGSC,2964,2964,True
17738,EC,57040,28165,False
17854,HGSC,37328,29531,False
18014,CC,58067,24234,False
18138,EC,29246,23480,False
18196,HGSC,64620,28043,False
18547,HGSC,43646,32109,False
18568,HGSC,51369,46414,False
18607,EC,44671,32875,False
18810,EC,70807,41434,False
18813,CC,54671,32443,False
18896,HGSC,56513,26126,False
18914,HGSC,61423,20688,False
18981,HGSC,29086,35820,False
19030,EC,17001,22962,False
19157,HGSC,66127,32663,False
19255,HGSC,64134,35246,False
19512,HGSC,90380,41650,False
19569,EC,47849,39194,False
20205,CC,34075,28265,False
20312,HGSC,61275,43983,False
20316,HGSC,48631,41367,False
20329,MC,67900,48795,False
20463,HGSC,47331,37823,False
20670,HGSC,63223,15903,False
20858,HGSC,40145,19260,False
20882,EC,37420,24860,False
21020,MC,3388,3388,True
21232,EC,41471,18923,False
21260,LGSC,61827,14268,False
21303,LGSC,53442,38926,False
21373,EC,42911,33671,False
21432,HGSC,48485,41454,False
21445,MC,34200,23705,False
21910,EC,71431,37669,False
21929,CC,46184,29963,False
22155,CC,42902,34751,False
22221,EC,36540,31231,False
22290,CC,60285,24009,False
22425,CC,73230,31569,False
22489,CC,26703,28946,False
22654,HGSC,75855,34323,False
22740,EC,32370,25928,False
22924,EC,58620,20295,False
23523,MC,74723,45387,False
23629,EC,41900,37320,False
23796,LGSC,41540,26909,False
24023,EC,65282,33033,False
24507,HGSC,31623,33203,False
24563,CC,60533,49543,False
24617,HGSC,34051,32507,False
24759,HGSC,72163,32003,False
24991,HGSC,33027,20740,False
25256,LGSC,47369,25545,False
25331,MC,61896,47117,False
25561,HGSC,54584,17787,False
25604,EC,31993,21712,False
25792,HGSC,37300,19280,False
25923,CC,27128,27946,False
25928,CC,33540,36394,False
26025,LGSC,52560,32700,False
26124,HGSC,29979,23288,False
26190,HGSC,66646,31600,False
26219,CC,55045,45975,False
26533,CC,28420,21151,False
26603,HGSC,93020,43000,False
26644,HGSC,69027,22189,False
26862,LGSC,79527,41540,False
26939,EC,40629,28433,False
26950,CC,35480,40363,False
27245,HGSC,46928,40170,False
27249,HGSC,27202,26827,False
27315,CC,37183,27885,False
27739,HGSC,80170,41514,False
27747,HGSC,47487,28122,False
27851,EC,68648,30510,False
27950,HGSC,57729,21360,False
28028,HGSC,48570,49215,False
28066,EC,45784,32200,False
28121,HGSC,73920,35995,False
28393,HGSC,31904,24382,False
28519,HGSC,39101,17164,False
28562,MC,65462,32887,False
28603,CC,61362,33866,False
28736,EC,18624,16747,False
28821,HGSC,33143,23882,False
28922,HGSC,40575,27243,False
29084,LGSC,3388,3388,True
29147,HGSC,38610,26181,False
29200,HGSC,50560,39023,False
29240,HGSC,60503,37691,False
29331,HGSC,72886,36720,False
29615,CC,36742,27683,False
29888,HGSC,21977,19338,False
29904,HGSC,74270,20498,False
29915,CC,68112,33540,False
30203,CC,55551,36311,False
30272,CC,35785,31903,False
30369,CC,66146,40994,False
30508,HGSC,75023,27968,False
30515,EC,77711,25227,False
30539,HGSC,58522,24345,False
30712,EC,48987,27960,False
30738,HGSC,65200,21728,False
30792,EC,64147,31531,False
30794,HGSC,52389,38513,False
30868,MC,62945,30028,False
30986,MC,66911,49395,False
31033,CC,44126,24065,False
31297,MC,67580,46491,False
31300,LGSC,22848,19232,False
31333,HGSC,65543,30023,False
31383,EC,53355,44443,False
31473,HGSC,43454,17267,False
31594,EC,3388,3388,True
31793,LGSC,18693,17541,False
32032,CC,47250,26631,False
32035,HGSC,16861,44075,False
32042,EC,30590,26403,False
32112,HGSC,45175,30035,False
32192,HGSC,17392,16832,False
32432,EC,35266,20507,False
32596,HGSC,70523,26526,False
32636,EC,80080,35135,False
33708,LGSC,47787,37629,False
33839,LGSC,28733,22106,False
33976,LGSC,38146,21220,False
33984,EC,43205,31323,False
34247,HGSC,46241,34689,False
34277,LGSC,65805,35570,False
34508,MC,29367,27783,False
34649,HGSC,34024,38850,False
34688,LGSC,27441,19507,False
34690,EC,31086,15615,False
34720,HGSC,34331,23131,False
34822,EC,43631,29642,False
34845,HGSC,42908,25840,False
35197,CC,55645,43515,False
35239,HGSC,61565,44551,False
35565,MC,2964,2964,True
35592,HGSC,61703,27422,False
35652,HGSC,80866,36163,False
35754,EC,35295,38271,False
35792,MC,82429,43195,False
35909,EC,48260,43380,False
35953,EC,70726,30286,False
35998,HGSC,30900,29909,False
36008,HGSC,54565,24542,False
36063,CC,83724,35889,False
36204,HGSC,65229,41831,False
36302,CC,3388,3388,True
36499,HGSC,29243,24399,False
36513,HGSC,81240,37483,False
36583,LGSC,3388,3388,True
36639,HGSC,31664,25529,False
36678,MC,76103,40490,False
36783,MC,2964,2964,True
37190,MC,83223,45871,False
37307,MC,46120,39107,False
37367,HGSC,39823,29565,False
37385,LGSC,3388,3388,True
37655,HGSC,56807,35254,False
38018,HGSC,37448,21486,False
38019,MC,59023,43223,False
38041,HGSC,64623,22387,False
38048,HGSC,55447,36535,False
38097,HGSC,78886,27688,False
38118,EC,77147,34046,False
38349,CC,45540,42755,False
38366,LGSC,31951,21718,False
38479,EC,58063,32383,False
38535,HGSC,45920,35052,False
38585,LGSC,64822,30320,False
38647,EC,40589,26593,False
38669,CC,41105,27185,False
38687,CC,61465,29965,False
38849,LGSC,60283,28011,False
38901,EC,53515,37187,False
38959,LGSC,58091,15650,False
39144,HGSC,55448,24327,False
39146,HGSC,27243,32722,False
39172,HGSC,45755,35007,False
39208,MC,61530,28220,False
39252,MC,48980,40700,False
39255,HGSC,62867,44147,False
39258,LGSC,47191,50155,False
39269,EC,29455,18346,False
39297,HGSC,73987,27220,False
39365,HGSC,54771,21407,False
39425,HGSC,68946,41260,False
39466,LGSC,45103,29828,False
39728,HGSC,43768,27620,False
39872,MC,55223,39867,False
39880,CC,40829,28620,False
39893,HGSC,57852,43654,False
39990,HGSC,28206,32663,False
40079,EC,29229,22081,False
40124,EC,55186,32887,False
40129,CC,61200,25520,False
40503,EC,61440,26709,False
40639,HGSC,24581,18896,False
40864,LGSC,2964,2964,True
40888,CC,76355,49135,False
41099,LGSC,20083,19155,False
41361,HGSC,78149,21344,False
41368,EC,3388,3388,True
41586,CC,2964,2964,True
41801,HGSC,81147,42934,False
42125,CC,43845,46047,False
42260,MC,36450,31720,False
42296,CC,69260,32815,False
42549,CC,72496,41725,False
42857,CC,2964,2964,True
43280,EC,79191,29689,False
43390,HGSC,56447,35154,False
43432,HGSC,34182,32040,False
43671,HGSC,40783,36475,False
43796,CC,70772,36940,False
43815,HGSC,38935,39255,False
43875,HGSC,74627,30260,False
43998,EC,64983,27706,False
44232,EC,77992,30533,False
44283,CC,20903,24764,False
44432,HGSC,27333,18732,False
44530,MC,53239,49293,False
44581,CC,30786,23591,False
44603,CC,3388,3388,True
44700,EC,36243,16723,False
44804,EC,56082,20080,False
44962,EC,67046,31872,False
44976,CC,67671,23860,False
45104,HGSC,68112,29826,False
45185,HGSC,50967,41307,False
45254,EC,62667,14444,False
45578,EC,71092,22588,False
45630,EC,65872,46075,False
45725,CC,68255,28430,False
45990,CC,63660,28000,False
46139,HGSC,33347,24842,False
46172,HGSC,42070,39450,False
46435,HGSC,46123,44540,False
46444,EC,32120,26182,False
46469,HGSC,56200,47455,False
46543,CC,70031,28015,False
46688,EC,52508,46680,False
46736,HGSC,39440,20045,False
46769,CC,69343,39330,False
46793,HGSC,71300,35403,False
46815,MC,40640,45411,False
47020,HGSC,71666,36620,False
47035,EC,2964,2964,True
47105,EC,39020,39353,False
47431,MC,67495,46563,False
47837,HGSC,35743,37460,False
47911,HGSC,45465,24590,False
47960,CC,51720,43711,False
47984,HGSC,47165,17046,False
48502,MC,44540,17300,False
48506,CC,101254,34003,False
48550,MC,32431,25393,False
48734,EC,3388,3388,True
48861,HGSC,41468,20360,False
48973,HGSC,70020,15640,False
49281,HGSC,45344,37340,False
49587,MC,59564,48608,False
49872,LGSC,73050,16818,False
49942,EC,23622,15342,False
49995,EC,25197,18134,False
50048,EC,40062,18080,False
50246,EC,60663,24740,False
50304,EC,52562,47435,False
50589,HGSC,34271,21730,False
50712,HGSC,67800,26380,False
50878,HGSC,41808,27147,False
50932,HGSC,2964,2964,True
50962,EC,59670,39962,False
51021,HGSC,72720,37900,False
51032,EC,91965,40838,False
51128,EC,59407,36435,False
51215,EC,8448,7590,False
51346,CC,84440,48363,False
51499,EC,43763,30691,False
51679,EC,38328,32683,False
51832,HGSC,75960,21282,False
51893,MC,53524,48275,False
52108,EC,16621,14700,False
52259,CC,73411,36474,False
52275,HGSC,21301,14880,False
52308,HGSC,29684,22073,False
52375,CC,47643,22007,False
52420,CC,64391,28803,False
52438,HGSC,33243,45695,False
52461,CC,56535,24491,False
52612,LGSC,54144,26303,False
52752,EC,72263,39683,False
52784,HGSC,25059,22084,False
52836,EC,44915,32840,False
52846,EC,60466,44891,False
52931,CC,58547,36923,False
53059,HGSC,70594,27442,False
53098,EC,36830,34000,False
53377,CC,36337,25394,False
53402,EC,30235,19720,False
53655,EC,3388,3388,True
53688,HGSC,26591,29871,False
53859,EC,52240,38383,False
53900,EC,61164,34452,False
54007,HGSC,21462,21749,False
54408,EC,48943,34547,False
54473,LGSC,63233,19880,False
54506,HGSC,50800,20620,False
54590,HGSC,56010,35066,False
54825,HGSC,55315,24726,False
54928,CC,36166,31487,False
54949,HGSC,37903,28174,False
54990,HGSC,76800,32800,False
55279,EC,66200,38227,False
55281,HGSC,68171,18203,False
55287,CC,47680,45400,False
55876,EC,28760,15866,False
56117,MC,64953,33716,False
56221,HGSC,31523,20020,False
56351,MC,52939,39618,False
56500,LGSC,65748,25123,False
56799,MC,51304,33963,False
56843,HGSC,71169,19205,False
56861,CC,45535,47147,False
56875,EC,46242,29086,False
56947,CC,59288,24926,False
56993,MC,51910,47471,False
57100,HGSC,51000,18220,False
57162,LGSC,43752,46643,False
57265,MC,32506,38820,False
57468,HGSC,40802,34003,False
57598,CC,40495,29775,False
57696,CC,3388,3388,True
57711,EC,60530,34049,False
58895,HGSC,40100,43843,False
58947,CC,58000,38350,False
58974,CC,68443,40383,False
59002,EC,35043,27707,False
59031,HGSC,48866,47340,False
59383,HGSC,39047,37211,False
59515,CC,64700,36387,False
59589,HGSC,49751,33607,False
59760,EC,59127,28531,False
59900,EC,57345,22048,False
60058,HGSC,28984,15447,False
60287,HGSC,57120,30447,False
60685,CC,74009,21583,False
60928,EC,102100,37720,False
60936,EC,47082,43243,False
60988,CC,44333,41947,False
61033,HGSC,69600,29311,False
61089,HGSC,32369,19660,False
61100,LGSC,40209,19886,False
61280,HGSC,43847,23594,False
61320,EC,36621,43071,False
61424,HGSC,44889,26787,False
61493,HGSC,33746,33340,False
61689,HGSC,65695,18220,False
61797,HGSC,3388,3388,True
61823,EC,32807,19314,False
61852,HGSC,33401,36180,False
61961,CC,47531,45240,False
62476,MC,91031,38474,False
62641,HGSC,16662,32055,False
62828,CC,62525,33406,False
63015,CC,52483,35320,False
63121,EC,53469,44300,False
63165,CC,30342,12783,False
63289,CC,32380,35307,False
63298,HGSC,26067,20341,False
63367,EC,62905,24783,False
63429,EC,67783,29066,False
63836,EC,17416,21934,False
63897,HGSC,9022,23582,False
63941,HGSC,47123,36600,False
64111,HGSC,15549,8129,False
64188,HGSC,77833,30683,False
64629,HGSC,25480,14920,False
64771,EC,29163,27940,False
64824,CC,46589,19365,False
64950,HGSC,70968,37971,False
65022,LGSC,53355,46675,False
65094,MC,55042,45080,False
65300,HGSC,75860,27503,False
65371,HGSC,42551,41800,False
65533,HGSC,45190,33980,False

In [ ]:
# 2 · inputs
import numpy as np, pandas as pd, torch
sys.path.insert(0, '/tmp/code')
import kfigs
from mil.models import build_model
OUT = '/kaggle/working/figures'; os.makedirs(OUT, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
PICK_IDS = ['1952', '22221', '10143', '11557', '2227', '33976']   # same slides as fig_heatmaps

metas = [os.path.dirname(m) for m in glob.glob('/kaggle/input/**/meta.json', recursive=True)]
roots = [r for r in metas if os.path.isdir(os.path.join(r, 'features', 'vits16_lunit')) and os.path.isdir(os.path.join(r, 'coords'))]
assert roots, 'features not attached: Add Input -> Your Work -> Notebooks -> notebook4297f7f927'
FEATS, COORDS = os.path.join(roots[0], 'features', 'vits16_lunit'), os.path.join(roots[0], 'coords')
cands = sorted(glob.glob('/kaggle/input/**/runs/vits16_lunit/abmil', recursive=True),
               key=lambda d: len(glob.glob(f'{d}/seed*/fold*/best.pt')), reverse=True)
assert cands and glob.glob(f'{cands[0]}/seed*/fold*/best.pt'), 'models not attached: Add Input -> notebookb9c62d68fa'
pngs = glob.glob('/kaggle/input/**/*.png', recursive=True)
num = lambda p: (re.match(r'(\d+)', os.path.basename(p)) or [None, None])[1]
FULL = {num(p): p for p in pngs if 'train_images' in p}
THUMBS = {num(p): p for p in pngs if 'train_thumbnails' in p}
assert FULL, 'slides not found: Add Input -> Competitions -> UBC-OCEAN (wait until the data banner finishes)'

a = pd.read_csv('/tmp/code/assignment.csv', dtype={'file_name': str}).rename(columns={'file_name': 'image_id'})
dims = pd.read_csv('/tmp/code/kaggle_train.csv', dtype={'image_id': str})[['image_id', 'image_width', 'image_height']]
SLIDES = a.merge(dims, on='image_id')
dim = np.load(glob.glob(os.path.join(FEATS, '*.npy'))[0], mmap_mode='r').shape[1]
MODELS = kfigs.load_models(cands[0], build_model, dim, device)
PICKS = SLIDES.set_index('image_id').loc[[i for i in PICK_IDS if i in FULL]].reset_index()
print(f'{len(MODELS)} models | {len(FULL)} slides, {len(THUMBS)} thumbnails | close-ups for {list(PICKS.image_id)}')

In [ ]:
# 3 · close-up figure
kfigs.closeup_figure(MODELS, PICKS, FEATS, COORDS, THUMBS, FULL, device, OUT)

In [ ]:
# 4 · download bundle
import zipfile
with zipfile.ZipFile('/kaggle/working/closeups_bundle.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f'{OUT}/*'):
        z.write(f, os.path.relpath(f, '/kaggle/working'))
print(f'closeups_bundle.zip: {os.path.getsize("/kaggle/working/closeups_bundle.zip") / 1e6:.1f} MB | {(time.time() - T0) / 60:.1f} min')